# What is an MCP?

Simply put, <br>
 **MCP is a protocol that standardizes how agents access tools and data.
MCP servers host the tools, and MCP clients connect agents to those servers.**
<br>
Here are some examples.
<br>

Lets say that you are a malicious CIA agent who is trying to overthrow a foreign government with hundreds of agentic AI's meant hack into a goverments datacenter, or you are consulting for LinkedIn.

<br>


In the CIA example, you could connect your agents to a MCP server so that all of your agents can have access to all the same tools and have a seperate computer run the tools instead of whatever the agents are being run on. In the LinkedIn example, LinkedIn can create a MCP server with tools they made so that they can control the flow of sensitive data being sent to your agents (this may actually be coming soon).

<br>

# How are MCP's different from tools?
<br>
A tool is some sort of logic that is executed by Python, in your file, on your computer. An MCP however exposes tools or in other words shows them to the agent. For example lets say you had 3 agents running on your computer for some reason and you want them all to use a tool. You can locally host a MCP that all the agents connect to and can run. An agent can be connected to multiple MCP servers, and MCP's can be run remotely or locally.

<br>
<br>
<br>
Here is a link to a very useful MCP walkthrough
<br>
<br>
https://dev-faizan.medium.com/building-your-first-mcp-server-with-python-in-2026-complete-tutorial-with-3-real-examples-49700e902656


# With that lets connect to our first MCP!





# Important: Below are two code chunks, you will need to copy and paste these chunks into two different files (in the same folder as a .env) and run them in two different terminals.

In [ ]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

In [ ]:
!pip install logging
!pip install os
!pip install random
!pip install sys
!pip install requests
!pip install mcp.server.fastmcp
!pip install dotenv
!pip install langchain_google_genai
!pip install langchain.agents
!pip install langchain_mcp_adapters.client

Here is the first code chunk that creates the MCP server

In [ ]:



import logging
import os
import random
import sys
import requests
from mcp.server.fastmcp import FastMCP


# Link for walkthrough
#  https://dev-faizan.medium.com/building-your-first-mcp-server-with-python-in-2026-complete-tutorial-with-3-real-examples-49700e902656



# this portion is optional but recommended for better logging and debugging of your server. It sets up a logger that you can use to output info, warnings, and errors in a structured way.
####################################
# Server name for logs
name = "demo-mcp-server"

# Configure logging output
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# Create logger instance
logger = logging.getLogger(name)
#####################################

# Get port from environment variable or use default
port = int(os.environ.get('PORT', 8080))

# Create MCP server instance
mcp = FastMCP(name, port=port)

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    logger.info(f"Tool called: add({a}, {b})")
    return a + b

# test result = add(10, 15)
#      print(result)

@mcp.tool()
def get_current_weather(city: str) -> str:
    """Get current weather for a city"""
    logger.info(f"Tool called: get_current_weather({city})")

    try:
        # Use wttr.in public weather service
        endpoint = "https://wttr.in"
        response = requests.get(f"{endpoint}/{city}", timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        logger.error(f"Error fetching weather data: {str(e)}")
        return f"Error fetching weather data: {str(e)}"


# test    weather = get_current_weather("Tokyo")
#         print(weather)


if __name__ == "__main__":
    logger.info(f"Starting MCP Server on port {port}...")
    try:
        mcp.run(transport="sse")
    except Exception as e:
        logger.error(f"Server error: {str(e)}")
        sys.exit(1)
    finally:
        logger.info("Server terminated")



Here is the code chunk for the next file that creates the agent and connects to the MCP

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

async def main():
    client = MultiServerMCPClient({
        "weather": {
            "transport": "sse",
            "url": "http://localhost:8080/sse", #<-- URL for the MCP server's SSE endpoint
        }
    })

    tools = await client.get_tools()  # <-- await here! async function to fetch tools from the MCP server

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"),
    )

    agent = create_agent(
        llm,
        tools=tools,
        system_prompt="You are a helpful assistant. Use the tools to answer user queries."
    )

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": "What’s the weather in Tokyo?"}]
    })
    print(response)

if __name__ == "__main__":
    asyncio.run(main())


# run these commands in terminal to start the servers in SEPERATE terminals and test the agent:
# python3 virtual_math_weather_server.py
# python3 math_mcp.py